# Statistics for ML — AI Lab Instructor Notebook

*Math for ML · Easy*

Master ML-relevant statistics: estimators, confidence intervals, hypothesis tests, p-values, multiple comparisons, and common gotchas.

**Outcome.** Students can compute and interpret confidence intervals and tests, understand failure modes (p-hacking, confounding), and connect stats to ML evaluation.

6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Statistics for ML — FAANG-Level Lab

**Goal:** Confidence intervals, hypothesis testing, and interpretation for ML engineering.

**Outcome:** You can quantify uncertainty and avoid p-value traps.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

**Why this works.** # Statistics for ML — Instructor (Solutions + Rationale)

## Task 2: Unbiased sample variance
*Section: Estimators (Mean/Variance)*

## Section 1 — Estimators (Mean/Variance)

### Task 1.1: Unbiased sample variance
Implement sample mean and unbiased sample variance (ddof=1) without calling np.var(..., ddof=1).

- mean = sum(x)/n
- unbiased var = sum((x-mean)^2)/(n-1)

**Explain:** Why divide by (n-1) instead of n?

**Hints**
- mean = sum(x)/n
unbiased var = sum((x-mean)^2)/(n-1)

**Explain:** Why divide by (n-1) instead of n?

In [ ]:
def sample_mean(x):
    x = np.asarray(x, dtype=float)
    return float(np.sum(x) / x.size)

def sample_var_unbiased(x):
    x = np.asarray(x, dtype=float)
    n = x.size
    m = sample_mean(x)
    return float(np.sum((x - m)**2) / (n - 1))

x = rng.standard_normal(1000)
check('mean_close', abs(sample_mean(x) - x.mean()) < 1e-10)
check('var_close', abs(sample_var_unbiased(x) - x.var(ddof=1)) < 1e-8)

# Rationale: dividing by (n-1) corrects bias because mean is estimated from data (loss of 1 DOF).

In [ ]:
# Checks
check('mean_close', abs(sample_mean(x) - x.mean()) < 1e-10)
check('var_close', abs(sample_var_unbiased(x) - x.var(ddof=1)) < 1e-8)

**Why this works.** dividing by (n-1) corrects bias because mean is estimated from data (loss of 1 DOF).

## Task 3: 95% CI for mean
*Section: Confidence Interval for Mean (Normal approx)*

## Section 2 — Confidence Interval for Mean (Normal approx)

### Task 2.1: 95% CI for mean
Compute a 95% CI for mean using normal approximation:
CI = mean ± z * s/sqrt(n), where z≈1.96.

- Use unbiased sample std

**FAANG gotcha:** CI is about the mean, not individual outcomes.

**Hints**
- Use unbiased sample std

In [ ]:
def mean_ci_normal(x, alpha=0.05):
    x = np.asarray(x, dtype=float)
    n = x.size
    m = x.mean()
    s = x.std(ddof=1)
    z = 1.96  # approx for 95%
    half = z * s / np.sqrt(n)
    return float(m - half), float(m + half)

x = rng.normal(2.0, 3.0, size=500)
lo, hi = mean_ci_normal(x)
check('order', lo < x.mean() < hi)

# Rationale: CI estimates plausible range for population mean around sample mean.

In [ ]:
# Checks
check('order', lo < x.mean() < hi)

**Why this works.** CI estimates plausible range for population mean around sample mean.

## Task 4: Coverage simulation
*Section: Confidence Interval for Mean (Normal approx)*

### Task 2.2: Coverage simulation
Simulate repeated sampling from Normal(mu=0, sigma=1).
Estimate how often 95% CI contains true mean.

- Run many trials
- Count coverage

**Explain:** Why isn't coverage exactly 0.95 in finite simulation?

**Hints**
- Run many trials
Count coverage

**Explain:** Why isn't coverage exactly 0.95 in finite simulation?

In [ ]:
def estimate_ci_coverage(trials=2000, n=50):
    mu = 0.0
    covered = 0
    for _ in range(trials):
        s = rng.normal(mu, 1.0, size=n)
        lo, hi = mean_ci_normal(s)
        covered += int(lo <= mu <= hi)
    return covered / trials

cov = estimate_ci_coverage(trials=2000, n=50)
print('coverage', cov)
check('reasonable', 0.92 < cov < 0.98)

# Rationale: coverage is ~0.95 by construction; simulation error causes small deviation.

In [ ]:
# Checks
check('reasonable', 0.92 < cov < 0.98)

**Why this works.** coverage is ~0.95 by construction; simulation error causes small deviation.

## Task 5: Permutation test for A/B (no scipy)
*Section: Hypothesis Testing (Two-sample test intuition)*

## Section 3 — Hypothesis Testing (Two-sample test intuition)

### Task 3.1: Permutation test for A/B (no scipy)
Given samples A and B, test whether mean(B) - mean(A) is significant via permutation.

- Combine samples
- Shuffle and split
- Compute diff distribution
- p-value = fraction of diffs >= observed (two-sided if needed)

**FAANG gotcha:** p-value is not P(H0 true).

**Hints**
- Combine samples
Shuffle and split
Compute diff distribution
p-value = fraction of diffs >= observed (two-sided if needed)

In [ ]:
def permutation_pvalue(A, B, n_perm=5000, two_sided=True):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    obs = B.mean() - A.mean()
    pooled = np.concatenate([A, B])
    nA = A.size
    diffs = np.empty(n_perm)
    for i in range(n_perm):
        perm = rng.permutation(pooled)
        diffs[i] = perm[nA:].mean() - perm[:nA].mean()
    if two_sided:
        p = np.mean(np.abs(diffs) >= abs(obs))
    else:
        p = np.mean(diffs >= obs)
    return float(p)

A = rng.normal(0.0, 1.0, size=200)
B = rng.normal(0.2, 1.0, size=200)
p = permutation_pvalue(A, B, n_perm=2000)
print('p-value', p)
check('p_range', 0 <= p <= 1)

# Rationale: permutation test is nonparametric; it uses the null distribution induced by exchangeability.

In [ ]:
# Checks
check('p_range', 0 <= p <= 1)

**Why this works.** permutation test is nonparametric; it uses the null distribution induced by exchangeability.

## Task 6: Bonferroni correction
*Section: Multiple Comparisons (Gotcha)*

## Section 4 — Multiple Comparisons (Gotcha)

### Task 4.1: Bonferroni correction
If you run m tests at alpha=0.05, Bonferroni uses alpha/m per test.

Compute adjusted alpha for m=20 and explain why this matters in feature slicing / metric dashboards.

In [ ]:
m = 20; alpha=0.05
alpha_adj = alpha / m
print('alpha_adj', alpha_adj)
check('alpha_adj', abs(alpha_adj - 0.0025) < 1e-12)

# Rationale: multiple comparisons inflate false positives; dashboards/slicing can create p-hacking unintentionally.

In [ ]:
# Checks
check('alpha_adj', abs(alpha_adj - 0.0025) < 1e-12)

**Why this works.** multiple comparisons inflate false positives; dashboards/slicing can create p-hacking unintentionally.